# 01 · Wrangling 11 invoice asli (DS 2, Minggu 2)

Tujuan: mengubah isi 11 invoice asli menjadi dua tabel bersih (`invoice_header` dan `invoice_lines`), memeriksanya, lalu menghitung metrik awal dari `docs/data_requirements_mapping.md`.

> **Data asli.** Invoice dan CSV transkripsinya ada di `data/raw/` dan hasil olahan di `data/processed/` (keduanya di-`.gitignore`). **Kosongkan semua output sebelum commit** (Kernel → Restart & Clear Output). Jangan menulis nama, HP, atau alamat pelanggan; pelanggan hanya `C01` dan `C02`.

Acuan: `docs/invoice_review_notes.md`, `docs/data_limitation_log.md` (L01–L08, asumsi A01–A07, definisi D01–D05), dan menu list KF bagian 3.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

HEADER_CSV = RAW / "invoice_header.csv"
LINES_CSV = RAW / "invoice_lines.csv"
for f in (HEADER_CSV, LINES_CSV):
    assert f.exists(), f"Belum ada {f}. Salin dari Drive tim atau isi templatenya."

# Menu list resmi (KF bagian 3), harga dalam rupiah.
MENU_PRICE = {"Mackster": 35000, "Nutrimac": 37000, "Wraphoria": 27000, "Mac Bites": 25000}
SHIPPING_PER_DELIVERY = 20000   # asumsi A02
EXPECTED_INVOICES = 11

## 1. Muat data

Dua CSV hasil transkripsi manual dari PDF (cek ulang tiap angka ke PDF-nya):

- `invoice_header.csv`: satu baris per invoice.
- `invoice_lines.csv`: satu baris per item. Isi `item_type` dengan `menu`, `bundling`, `bonus`, atau `kolaborasi`.

Tanggal ditulis ISO (`YYYY-MM-DD`), uang dalam rupiah tanpa titik, bagian yang tidak tercetak di invoice dikosongkan (jangan ditebak).

In [ ]:
header = pd.read_csv(HEADER_CSV, parse_dates=["invoice_date"])
lines = pd.read_csv(LINES_CSV, parse_dates=["delivery_date"])
print(header.shape, lines.shape)
header.dtypes

## 2. Pembersihan

In [ ]:
# Samakan nama menu (temuan F10: "Macbites" di invoice vs "Mac Bites" di menu list).
NAME_FIX = {"Macbites": "Mac Bites", "MacBites": "Mac Bites"}
lines["product"] = lines["product"].str.strip().replace(NAME_FIX)
lines["variant"] = lines["variant"].fillna("").str.strip()

# Pecah revisi dari nomor invoice, mis. "INV/20260912/WPH/007-v3".
rev = header["invoice_no"].str.extract(r"^(?P<base>.+?)(?:-v(?P<rev>\d+))?$")
header["invoice_base"] = rev["base"]
header["revision"] = pd.to_numeric(rev["rev"]).fillna(0).astype(int)

# Jumlah pengiriman: eksplisit di invoice bila ada, selain itu tersirat dari ongkir (A02).
header["n_delivery"] = header["n_delivery_explicit"].fillna(header["n_delivery_implied"])

# Kelompok menu: hanya 4 menu resmi bertipe 'menu'; sisanya 'Lainnya' (bundling, bonus, kolaborasi, item bebas).
is_menu = lines["item_type"].eq("menu") & lines["product"].isin(MENU_PRICE)
lines["menu"] = lines["product"].where(is_menu, "Lainnya")

# Minggu Senin-Minggu dari tanggal invoice (A01, D02).
header["week_start"] = header["invoice_date"] - pd.to_timedelta(header["invoice_date"].dt.dayofweek, unit="D")
header[["invoice_no", "invoice_date", "week_start", "customer_code", "n_delivery"]]

## 3. Validasi

Ini bukti pembacaan sudah benar. Kalau ada yang gagal, periksa transkripsi ke PDF, bukan mengubah asumsi.

Angka kontrol dari `docs/invoice_review_notes.md` bagian 2: subtotal **Rp2.561.000**, ongkir **Rp300.000**, grand total **Rp2.861.000**.

In [ ]:
def check(cond, msg):
    print(("OK    " if cond else "GAGAL ") + msg)
    assert cond, msg

check(len(header) == EXPECTED_INVOICES, f"Ada {EXPECTED_INVOICES} invoice (terbaca {len(header)})")
check(header["invoice_no"].is_unique, "Nomor invoice unik")
check(set(lines["invoice_no"]) == set(header["invoice_no"]), "Setiap invoice punya baris item dan sebaliknya")
check(lines["item_type"].isin(["menu", "bundling", "bonus", "kolaborasi"]).all(), "item_type valid")

# Aritmetika per baris dan per invoice
check((lines["qty"] * lines["unit_price"] == lines["line_total"]).all(), "qty x harga = total baris")
sum_lines = lines.groupby("invoice_no")["line_total"].sum()
h = header.set_index("invoice_no")
check((sum_lines.reindex(h.index) == h["subtotal"]).all(), "Jumlah baris = subtotal")
check((h["subtotal"] + h["shipping"] == h["grand_total"]).all(), "Subtotal + ongkir = grand total")

# Nomor invoice memuat tanggal yang sama dengan field tanggal
dated = header["invoice_no"].str.extract(r"INV/(\d{8})/")[0]
check((pd.to_datetime(dated, format="%Y%m%d") == header["invoice_date"]).all(), "Tanggal di nomor = field tanggal")

# Ongkir konsisten dengan jumlah pengiriman (A02), kecuali yang gratis
paid = header[header["shipping"] > 0]
check((paid["shipping"] == paid["n_delivery"] * SHIPPING_PER_DELIVERY).all(), "Ongkir = jumlah pengiriman x Rp20.000")

# Angka kontrol DS 1
check(header["subtotal"].sum() == 2_561_000, "Total subtotal Rp2.561.000")
check(header["shipping"].sum() == 300_000, "Total ongkir Rp300.000")
check(header["grand_total"].sum() == 2_861_000, "Total grand total Rp2.861.000")

## 4. Metrik awal

Semua angka di sini adalah **omzet ditagih**, bukan omzet: status bayar hanya diketahui untuk MYM-KPZ (L02). Pertanyaan tentang catatan kustom (L03), jadwal ulang/batal (L01), dan tagihan belum dibayar (L02) **tidak dijawab dari invoice**.

In [ ]:
# 4a. Omzet ditagih per menu (tanpa ongkir, A04). Bonus Rp0 ikut dihitung qty-nya, omzet 0.
per_menu = (lines.groupby("menu")
            .agg(qty=("qty", "sum"), omzet=("line_total", "sum"))
            .sort_values("omzet", ascending=False))
per_menu["share_pct"] = (100 * per_menu["omzet"] / per_menu["omzet"].sum()).round(1)
per_menu

In [ ]:
# 4b. Omzet ditagih per minggu (Senin-Minggu, dari tanggal invoice: A01)
per_week = (header.groupby("week_start")
            .agg(invoice=("invoice_no", "count"), subtotal=("subtotal", "sum"), ongkir=("shipping", "sum")))
per_week

In [ ]:
# 4c. Ongkir, jumlah pengiriman, nilai rata-rata invoice
summary = {
    "total_subtotal": int(header["subtotal"].sum()),
    "total_ongkir": int(header["shipping"].sum()),
    "total_grand_total": int(header["grand_total"].sum()),
    "jumlah_pengiriman_tercatat": int(header["n_delivery"].sum()),
    "rata2_nilai_invoice": round(header["grand_total"].mean()),
}
summary

In [ ]:
# 4d. Konsentrasi pelanggan
by_customer = (header.groupby("customer_code")
               .agg(invoice=("invoice_no", "count"), subtotal=("subtotal", "sum"), grand_total=("grand_total", "sum")))
by_customer["share_subtotal_pct"] = (100 * by_customer["subtotal"] / by_customer["subtotal"].sum()).round(1)
by_customer["share_grand_total_pct"] = (100 * by_customer["grand_total"] / by_customer["grand_total"].sum()).round(1)
by_customer

In [ ]:
# 4e. Item bonus Rp0 dan kesesuaian harga dengan menu list (KF 3)
bonus = lines[lines["item_type"] == "bonus"][["invoice_no", "product", "qty", "unit_price"]]
menu_rows = lines[lines["item_type"] == "menu"].copy()
menu_rows["harga_menu_list"] = menu_rows["product"].map(MENU_PRICE)
menu_rows["cocok"] = menu_rows["unit_price"] == menu_rows["harga_menu_list"]
price_check = menu_rows.groupby("product").agg(baris=("cocok", "size"), cocok=("cocok", "sum"))
display(bonus)
price_check

In [ ]:
# 4f. Porsi per tanggal kirim: hanya baris yang punya tanggal kirim (L04). Sebagian saja, jangan disimpulkan.
per_delivery_date = (lines.dropna(subset=["delivery_date"])
                     .groupby("delivery_date")["qty"].sum())
per_delivery_date

## 5. Simpan hasil olahan

Disimpan ke `data/processed/` (di-`.gitignore`, **tidak di-commit** karena berasal dari invoice asli).

In [ ]:
header.to_csv(PROCESSED / "invoice_header_clean.csv", index=False)
lines.to_csv(PROCESSED / "invoice_lines_clean.csv", index=False)
per_week.to_csv(PROCESSED / "invoice_omzet_per_minggu.csv")
per_menu.to_csv(PROCESSED / "invoice_omzet_per_menu.csv")
print("Tersimpan di", PROCESSED)

## 6. Keterbatasan yang harus ikut dibaca

- 10 dari 11 invoice satu pelanggan (C01) dan satu menu (Wraphoria), jadi ranking menu dan tren **belum bermakna** (L08).
- Tanggal kirim per baris hanya ada di WPH/003 (L04); minggu memakai tanggal invoice (A01).
- Tidak ada field diskon (L05) dan bundling tidak dipecah (L06).
- Nomor 009 dan revisi v1/v2 dari WPH/007 tidak ada (L07).
- Angka agregat boleh masuk laporan hanya setelah dikonfirmasi tim; file olahan tidak di-commit.